<a href="https://colab.research.google.com/github/ankitsingh435517/lichess-gpt/blob/main/lichess_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!sudo apt-get install zstd
!pip install chess

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (580 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122809 files and director

In [103]:
!wget -qO- https://database.lichess.org/standard/lichess_db_standard_rated_2026-08.pgn.zst | zstd -d | head -n 1000 > games.pgn

In [104]:
!wc -l games.pgn

1000 games.pgn


In [105]:
import chess.pgn

games = []
with open('games.pgn', 'r', encoding='utf-8') as f:
  while game := chess.pgn.read_game(f):
    moves = []
    board = game.board()

    for move in game.mainline_moves():
      moves.append(board.san(move))
      board.push(move)

    if moves:
      games.append("".join(moves))

In [106]:
games

['c3d5d3c6Nd2Nf6Qc2Bf5e4dxe4dxe4Bg6Ngf3e6Be2Bd6a4O-OO-ONbd7Re1Qc7Bf1e5b4Rfe8Bc4a6Nf1b5axb5axb5Rxa8Rxa8Bb3Re8Bg5h6Bxf6Nxf6Nh4Bh7Nf5Bxf5exf5c5bxc5Qxc5Ng3b4c4Qd4Rd1Qb6Re1Bc5Ba4Rd8Bb5Ng4Rf1Nf6Re1e4Re2Rd4Nxe4Nxe4Rxe4Rxe4Qxe4b3Qe1Bd4Qb1b2Kf1Qc5Ke2Qb4Kd3Qb3+Kxd4Qb4Kd5Qd2+Kc5Qxf2+',
 'e4c5Nc3Nc6Nf3g6Bc4Bg7d3d6O-ONf6Ng5O-OQf3h6Qh3Bxh3Nxh3Nd4Be3Nxc2Rac1Nxe3fxe3e6Bb5a6Rf2axb5Rcf1Nd7Nf4e5',
 'e4e5Nc3Nc6Nf3Bc5d3d6Bd2Nge7Be2O-OO-ONg6a3f5b4Bb6exf5Bxf5Nd5Bg4Nh4Bxe2Qxe2Qxh4Nxb6axb6g3Qd8c4Nd4Qe3Nc2Qe4Nxa1Qxb7Nb3Qd5+Kh8c5Nxd2Rd1Nf3+Qxf3Rxf3Kg2Qf6Re1Rxf2+Kh3Qf5+g4Qf3#',
 'e4',
 'd4d5c4Nc6Nc3dxc4Nf3Nf6Bg5e6e3Bb4Bxc4h6Bh4g5Bg3g4Ne5Ne4O-ONxc3bxc3Bxc3Nxc6bxc6Rc1Bb4Be5f6Bg3h5Qa4h4Bf4Bd6Bxd6Qxd6Rb1g3h3Rg8Rbc1e5Bxg8gxf2+Rxf2exd4Qxc6+Qxc6Rxc6Bb7Rcxf6Ke7exd4Rxg8Rf7+Ke6R2f6+Kd5Rxc7Rxg2+',
 'e4d5exd5Qxd5Nc3Qa5Nf3c6Be2Nf6O-OBg4d4e6h4Be7Bg5h6Bxf6Bxf6Qd2Nd7Ne5Bxe2Qxe2Nxe5dxe5Qxe5Qxe5Bxe5Rfe1Bxc3bxc3O-O-ORed1Rxd1+Rxd1Rd8Re1Rd5Re4Rc5c4Ra5Rg4g6h5Rxh5',
 'b3e6Bb2d5Nf3Nc6e3Nf6Bb5Bd7Nc3a6Bxc6Bxc6d4Be7Qe2h6O-O-OQd7h3O-O-ONe5Q

In [107]:
# build vocab
text = "".join(games)
chars = ['.'] + sorted(set(text))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode: lambda

{'.': 0,
 '#': 1,
 '+': 2,
 '-': 3,
 '1': 4,
 '2': 5,
 '3': 6,
 '4': 7,
 '5': 8,
 '6': 9,
 '7': 10,
 '8': 11,
 '=': 12,
 'B': 13,
 'K': 14,
 'N': 15,
 'O': 16,
 'Q': 17,
 'R': 18,
 'a': 19,
 'b': 20,
 'c': 21,
 'd': 22,
 'e': 23,
 'f': 24,
 'g': 25,
 'h': 26,
 'x': 27}

In [117]:
# compile dataset into train, eval and test split - 90, 5, 5 split
# the spit takes 90% of data to train due to the fact that the whole dataset is roughly 5000 games
# so 250 games to validate and test is sufficient enough and we need atleast 4000 games to train
import random
import torch

# shuffle the games to make the dataset represent overall structures of the games and avoid certain kinds of biased structures if there exists example similar openings in first few games etc
random.shuffle(games)

n1 = int(0.9*(len(games))) # 90%
n2 = int(0.95*len(games)) # 95%

context_size = 3

def compile_dataset(games):
  # compile the data into X and Y (X is the input sequence & Y is the corresponding output sequence)
  # we need context length of input for which output has to be stored e.g 3 for now so 3 moves input then we should expect 4th move as output
  X, Y = [], []
  for game in games:
    context = [0] * context_size # 0 0 0 -> meaning a '.' representing no move as in start of the game.
    for move in game + ".":
      ix = stoi[move]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]

  return X, Y

def split_dataset(split="train"):
  # split
  data = {
      'train': games[:n1],
      'dev': games[n1:n2],
      'test': games[n2:]
  }[split]
  # compile
  X, Y = compile_dataset(data)
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

Xtr, Ytr = split_dataset("train")
Xdev, Ydev = split_dataset("dev")
Xte, Yte = split_dataset("test")
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)
print(Xte.shape, Yte.shape)

torch.Size([8521, 3]) torch.Size([8521])
torch.Size([382, 3]) torch.Size([382])
torch.Size([754, 3]) torch.Size([754])


In [102]:
0.9 * 4

3.6

In [ ]:
# tokenization - create a function which tokenizes into character level or BPE # train on both and check what is the different in training efficiency